# Chapter 12 — Faint and deep: the survey ladder

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/halo-substructure-tutorial/blob/main/notebooks/12_survey_ladder.ipynb)

**Goal.** The same satellite can be invisible in one survey and obvious in another. We take three faint satellites —
**Leo IV** ($M_V = -5.0$, 151 kpc, found in SDSS by Belokurov et al. 2007), **Virgo I** ($M_V = -0.9$, ~90 kpc) and
**Boötes IV** ($M_V = -5.3$, ~210 kpc), the last two found in HSC (Homma et al. 2016,
https://ui.adsabs.harvard.edu/abs/2016ApJ...832...21H; Homma et al. 2019,
https://ui.adsabs.harvard.edu/abs/2019PASJ...71...94H) — and search for each in four surveys of increasing depth: SDSS,
Pan-STARRS1, the Legacy Surveys DR9 and HSC-SSP PDR3.

**You will learn**
1. how depth and star–galaxy separation change what a survey can find;
2. to run one search on several surveys, each with its own error model and its own null calibration;
3. that "deeper" is not automatically "more significant" (more faint galaxies, different footprints, different classifiers).

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter        # smoothing of 2-d histograms (CMDs and sky maps)
from scipy import special, stats                 # incomplete gamma function (Poisson tail) and Gaussian quantiles

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are small extracts of survey catalogues, attached to a GitHub release of the tutorial.
# MANIFEST.json lists, for each file, the SQL that made it and its SHA-256 checksum.
DATA_URL = 'https://github.com/vasilybelokurov/halo-substructure-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release (again if the local copy is stale), check its SHA-256."""
    path = cache / f'{name}.parquet'
    if not path.exists():                          # first use: fetch the file into ./data
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != MANIFEST[name]['sha256']:   # stale or damaged copy
        print(f'downloading {path.name} again (checksum mismatch) ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    # the checksum guarantees that everyone runs this chapter on exactly the same rows
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST[name]['sha256'], f'{path} does not match the manifest'
    return pd.read_parquet(path)


def tangent(ra, dec, ra0, dec0):
    """Gnomonic (tangent-plane) projection about (ra0, dec0). Returns (xi, eta) in degrees: xi towards east, eta north."""
    # Flat coordinates around the target: over a few degrees the distortion is small, and unlike (ra, dec)
    # they do not stretch the sky by 1/cos(dec), so circles on the sky stay circles in (xi, eta).
    ra, dec, ra0, dec0 = map(np.radians, (ra, dec, ra0, dec0))
    cosc = np.sin(dec0) * np.sin(dec) + np.cos(dec0) * np.cos(dec) * np.cos(ra - ra0)   # cos(angular distance)
    xi = np.cos(dec) * np.sin(ra - ra0) / cosc
    eta = (np.cos(dec0) * np.sin(dec) - np.sin(dec0) * np.cos(dec) * np.cos(ra - ra0)) / cosc
    return np.degrees(xi), np.degrees(eta)


lvdb_mw = load('lvdb_mw').set_index('key')       # LVDB v1.1.1, Milky Way satellites: catalogued positions

## 12.1 Four catalogues of the same sky

Each file holds what that survey calls a point source, within 0.75° of the satellite, down to roughly its depth:

| Survey | Point sources | Limit | Isochrones used |
|---|---|---|---|
| SDSS DR9 | `type = 6` | g < 23 | PARSEC, SDSS bands |
| PS1 DR1 | $r_{\rm PSF} - r_{\rm Kron} < 0.05$ | g < 23.5 | PARSEC, PS1 bands |
| Legacy Surveys DR9 | Tractor `type = 'PSF'` | g < 24.5 | PARSEC, DECam (DES) bands |
| HSC-SSP PDR3 | `i_extendedness_value = 0` | g < 25.5 | PARSEC, SDSS bands (HSC g, r are close to SDSS; an approximation) |

Extinction: SDSS and the Legacy Surveys store their own per-object values (`extinction_*`, `mw_transmission_*`). For PS1
and HSC we use the median Legacy Surveys extinction of the same patch; at these high latitudes $A_g \approx 0.03$–0.1, and
the band-to-band differences are below 0.01 mag. Leo IV and Virgo I are in the southern (DECam) part of the Legacy
Surveys; Boötes IV (Dec 43.7°) is in the northern part (BASS/MzLS), whose filters differ slightly from DECam's — another
approximation.

In [ ]:
objects = {'leo4': 'leo_4', 'virgo1': 'virgo_1', 'bootes4': 'bootes_4'}   # data-file prefix -> LVDB key
SURVEYS = ['sdss', 'ps1', 'ls', 'hsc']                                    # in order of increasing depth
# isochrone set used for each survey (table above): LS uses the DECam (DES) bands; HSC uses SDSS bands (approximation)
ISO = {'sdss': 'isochrones_sdss', 'ps1': 'isochrones_ps1', 'ls': 'isochrones_des', 'hsc': 'isochrones_sdss'}
LIMIT = {'sdss': 23.0, 'ps1': 23.5, 'ls': 24.5, 'hsc': 25.5}            # faint g limit of each file [mag]
# Legacy Surveys fluxes (nanomaggies) -> AB magnitudes; NaN for non-positive fluxes
lsmag = lambda f: 22.5 - 2.5 * np.log10(np.where(f > 0, f, np.nan))


def photometry(obj, survey):
    """Dereddened g0, r0 and the g error of the point sources of one patch.

    Returns ra, dec [deg], g0, r0 and the g-band error [mag] as float arrays (NaN where a magnitude is undefined).
    """
    d = load(f'{obj}_{survey}')
    ls = load(f'{obj}_ls')
    # median Legacy Surveys extinction of the patch, A = -2.5 log10(transmission) [mag]; used for PS1 and HSC, whose
    # files carry no extinction (at these latitudes it varies little across a 0.75-deg patch)
    Ag_ls = np.median(-2.5 * np.log10(ls.mw_transmission_g)); Ar_ls = np.median(-2.5 * np.log10(ls.mw_transmission_r))
    if survey == 'sdss':
        # SDSS: PSF magnitudes minus the per-object extinction stored in the catalogue
        g, r, eg = d.psfmag_g - d.extinction_g, d.psfmag_r - d.extinction_r, d.psfmagerr_g
    elif survey == 'ps1':
        g, r, eg = d.gpsfmag - Ag_ls, d.rpsfmag - Ar_ls, d.gpsfmagerr
    elif survey == 'ls':
        g = lsmag(d.flux_g) + 2.5 * np.log10(d.mw_transmission_g)            # m0 = m + 2.5 log10(T)
        r = lsmag(d.flux_r) + 2.5 * np.log10(d.mw_transmission_r)
        eg = 1.0857 / (d.flux_g * np.sqrt(d.flux_ivar_g))                    # sigma_m = (2.5 / ln 10) sigma_f / f
    else:
        g, r, eg = d.g_psfflux_mag - Ag_ls, d.r_psfflux_mag - Ar_ls, d.g_psfflux_magerr
    return d.ra.to_numpy(), d.dec.to_numpy(), np.asarray(g, float), np.asarray(r, float), np.asarray(eg, float)


# Error law of each survey, measured in the Virgo I patch: median g error in 0.25-mag bins of g0 from 16 to the file's
# limit (bins with more than 5 stars only). Stored in depths[survey] = (bin centres, median errors) for the CMD masks.
depths = {}
fig, ax = plt.subplots(figsize=(7, 4))
for s in SURVEYS:
    _, _, g, r, eg = photometry('virgo1', s)
    bins = np.arange(16, LIMIT[s] + 0.01, 0.25)
    mid = 0.5 * (bins[1:] + bins[:-1])
    med = np.array([np.nanmedian(eg[(g >= a) & (g < b)]) if ((g >= a) & (g < b)).sum() > 5 else np.nan for a, b in zip(bins[:-1], bins[1:])])
    ok = np.isfinite(med) & (med > 0)                # drop empty bins (and zero errors, which a log axis cannot show)
    depths[s] = (mid[ok], med[ok])
    ax.semilogy(mid[ok], med[ok], 'o-', ms=3, label=f'{s.upper()} ({len(g):,} point sources)')
ax.axhline(0.1086, c='k', ls=':', lw=0.8)            # error of 0.1086 mag = S/N 10 (1.0857 / 10)
ax.set(xlabel='$g_0$', ylabel='median error in g', title='Virgo I patch: photometric errors')
ax.legend(fontsize=8)
plt.tight_layout()

**Figure 12.1.** Photometric depth of the four surveys in the Virgo I patch (point sources within 0.75°): median g-band
error (log scale) against extinction-corrected $g_0$, in 0.25-mag bins with more than 5 stars, for SDSS DR9 (blue,
8,760 point sources), PS1 DR1 (orange, 13,534), the Legacy Surveys DR9 (green, 28,537) and HSC-SSP PDR3 (red, 12,126).
The dotted line is an error of 0.1086 mag, i.e. S/N = 10. S/N = 10 is reached at g ≈ 22.4 in SDSS and ~22.7 in PS1 (with
the strict PSF − Kron cut), ~24 in the Legacy Surveys and beyond 25 in HSC (whose errors stay below 0.07 mag down to the
file limit). Note also the number of point sources: deeper data contain more faint galaxies classified as stars
(Chapter 1), and HSC's classifier rejects most of them.

## 12.2 The search of Chapter 5, survey by survey

For each survey we build the CMD masks with *its* isochrones and *its* measured errors, search the patch (kernels 1′, 2′
and 4′; background kernel 15′, appropriate for a 0.75° patch; distance moduli 18.5–22.5), and calibrate with 50 null
patches of the same survey (stars placed uniformly, Chapter 5). We record the largest S within 3′ of the catalogued
position, and the 98th percentile of the null maxima.

In [ ]:
MUS = np.arange(18.5, 22.51, 0.5)                # trial distance moduli: 18.5-22.5 in steps of 0.5 (50-316 kpc)
SIGMAS = np.array([1, 2, 4])                     # kernel widths in map pixels of 1', i.e. 1', 2', 4'
cb = np.arange(-0.4, 1.21, 0.04)                 # colour bin edges (g-r)_0: -0.4 to 1.2 in 0.04 mag (40 bins)
cc = 0.5 * (cb[1:] + cb[:-1])


def masks_for(survey, g0, c0):
    """CMD masks for every trial distance modulus, with the survey's isochrones and error law.

    Returns a float array of shape (len(MUS), N): 1 if star n is inside the mask for distance modulus MUS[k], else 0.
    The model and mask are those of Chapters 4-5 (and satellite_cmd / cmd_mask in Chapter 11).
    """
    mb = np.arange(16, LIMIT[survey] + 0.001, 0.1)   # magnitude bin edges: 0.1 mag, down to the survey's limit
    mm = 0.5 * (mb[1:] + mb[:-1])
    CC, MM = np.meshgrid(cc, mm, indexing='ij')
    iso = load(ISO[survey])
    # old, metal-poor population: 12.6 Gyr, [Fe/H] = -2.1, PARSEC stages up to the early AGB
    I = iso[np.isclose(iso.logage, 10.1) & np.isclose(iso.feh, -2.1) & (iso.stage <= 7)].sort_values('m_ini')
    c, g = (I.g - I.r).to_numpy(), I.g.to_numpy()
    # 3000 points equally spaced in CMD arc length, weighted by the fraction of stars from the IMF
    arc = np.concatenate([[0], np.cumsum(np.hypot(np.diff(c), np.diff(g)))])
    s = np.linspace(0, arc[-1], 3000)
    pc, pgi, w = np.interp(s, arc, c), np.interp(s, arc, g), np.gradient(np.interp(s, arc, I.int_imf))
    em, ee = depths[survey]
    # measured median errors; beyond the last measured magnitude they grow by a factor e per magnitude
    err = lambda m: np.interp(m, em, ee) * np.exp(np.clip(m - em[-1], 0, 5))
    # background CMD density from all stars of the patch, smoothed by one bin, floored at 1 count per cell
    H = gaussian_filter(np.histogram2d(c0, g0, [cb, mb])[0], 1.0)
    B = np.maximum(H, 1.0) / np.maximum(H, 1.0).sum()
    out = []
    for mu in MUS:
        pg = pgi + mu                                # isochrone at this distance modulus
        # Gaussian widths in colour and magnitude, with floors of 0.03 and 0.1 mag in quadrature
        sc, sm = np.hypot(np.sqrt(2) * err(pg), 0.03), np.hypot(err(pg), 0.1)
        S = np.einsum('k,kij->ij', w, np.exp(-0.5 * ((CC - pc[:, None, None]) / sc[:, None, None])**2
                                             - 0.5 * ((MM - pg[:, None, None]) / sm[:, None, None])**2) / sc[:, None, None] / sm[:, None, None])
        S /= S.sum()                                 # satellite CMD density on the grid, sums to 1
        # mask: cells in decreasing S/B, cut where signal / sqrt(background) peaks among masks with >= 50% of the signal
        order = np.argsort((S / B).ravel())[::-1]
        cs, cbk = np.cumsum(S.ravel()[order]), np.cumsum(B.ravel()[order])
        M = np.zeros(S.size, bool)
        M[order[:np.argmax(np.where(cs >= 0.5, cs / np.sqrt(cbk), 0)) + 1]] = True
        # look up each star's CMD cell; stars outside the grid are not in the mask
        i, j = np.digitize(c0, cb) - 1, np.digitize(g0, mb) - 1
        ok = (i >= 0) & (i < len(cc)) & (j >= 0) & (j < len(mm))
        out.append(ok & M.reshape(S.shape)[np.clip(i, 0, len(cc) - 1), np.clip(j, 0, len(mm) - 1)])
    return np.array(out, float)


def search(x, y, w, mask, extent, pix=1 / 60, sig_bg=15):
    """Chapter 5 search: maximum S over (mu, sigma) per pixel.

    x, y: star positions [deg]; w: mask weights from masks_for, shape (len(MUS), N); mask: boolean footprint on the
    pixel grid; pix: pixel size [deg] (1'); sig_bg: width of the background kernel [pixels] (15').
    Returns the map of the largest S over distance moduli and kernels, 0 where there is no overdensity.
    """
    xe, ye = np.arange(extent[0], extent[1] + pix / 2, pix), np.arange(extent[2], extent[3] + pix / 2, pix)
    M = mask.astype(float)
    norm_bg = gaussian_filter(M, sig_bg)             # fraction of the background kernel inside the footprint
    best = np.zeros(mask.shape)
    for wa in w:                                     # one distance modulus at a time
        W = np.histogram2d(x, y, [xe, ye], weights=wa)[0]           # masked stars per pixel
        Wb = gaussian_filter(W, sig_bg) / np.maximum(norm_bg, 1e-9)  # local background per pixel, edge-corrected
        for s in SIGMAS:
            norm = gaussian_filter(M, s)             # fraction of the kernel inside the footprint
            area = 4 * np.pi * s**2 * norm           # effective area of the Gaussian kernel [pixels] (Chapter 5)
            n = gaussian_filter(W, s) / np.maximum(norm, 1e-9) * area   # effective number of stars in the kernel
            lam = np.maximum(Wb * area, 1e-9)        # expected background in the same area
            # Poisson tail P(N >= n | lam) = regularised lower incomplete gamma P(n, lam), as a Gaussian-equivalent S;
            # clipped at 1e-300 so that S stays finite for very large excesses
            S = stats.norm.isf(np.clip(special.gammainc(np.maximum(n, 1e-9), lam), 1e-300, 1))
            # keep only overdensities (n > lam) in pixels with at least half of the kernel inside the footprint
            best = np.maximum(best, np.where(mask & (norm > 0.5) & (n > lam), S, 0.0))
    return best


rmax = 0.74                                      # search radius [deg], just inside the 0.75-deg radius of the files
extent = (-rmax, rmax, -rmax, rmax)
pc_ = np.arange(-rmax, rmax, 1 / 60) + 1 / 120   # pixel centres [deg] of the 1' grid
PX, PY = np.meshgrid(pc_, pc_, indexing='ij')
footprint = np.hypot(PX, PY) < rmax
inner = np.hypot(PX, PY) < rmax - 0.25           # at least 0.25 deg (= the 15' background kernel) from the edge
near = np.hypot(PX, PY) < 3 / 60                 # within 3' of the catalogued position
rng = np.random.default_rng(12)                  # fixed seed: the null patches are reproducible
rows, maps = [], {}
for obj, key in objects.items():
    ref = lvdb_mw.loc[key]                                        # all Milky Way satellites, LVDB v1.1.1
    for s in SURVEYS:
        ra, dec, g, r, eg = photometry(obj, s)
        x, y = tangent(ra, dec, ref.ra, ref.dec)                  # [deg], centred on the catalogued position
        keep = np.isfinite(g) & np.isfinite(r) & (np.hypot(x, y) < rmax)
        x, y, g, c = x[keep], y[keep], g[keep], (g - r)[keep]
        w = masks_for(s, g, c)
        Smap = search(x, y, w, footprint, extent)
        maps[obj, s] = Smap
        # Null: 50 patches with the same stars (magnitudes, colours, mask weights) placed uniformly in the disc
        # (r = rmax sqrt(u) gives a uniform surface density); record the largest S of each away from the edge
        null = []
        for t in range(50):
            rr, th = rmax * np.sqrt(rng.uniform(0, 1, len(x))), rng.uniform(0, 2 * np.pi, len(x))
            null.append(search(rr * np.cos(th), rr * np.sin(th), w, footprint, extent)[inner].max())
        # with 50 null maxima, the 98th percentile lies between the two largest
        rows.append(dict(object=key, survey=s.upper(), S_at_object=Smap[near].max(), null_98=np.percentile(null, 98),
                         max_elsewhere=Smap[inner & ~near].max()))
res = pd.DataFrame(rows)
res['detected'] = res.S_at_object > res.null_98  # detection: the satellite beats 98% of that survey's null maxima
print(res.round(2).to_string(index=False))

In [ ]:
# 3 x 4 grid of search maps: rows = satellites, columns = surveys; S from 0 (black) to 8 (white, saturated above)
fig, ax = plt.subplots(3, 4, figsize=(14, 10.5))
for a_, (obj, key) in enumerate(objects.items()):
    for b_, s in enumerate(SURVEYS):
        row = res[(res.object == key) & (res.survey == s.upper())].iloc[0]
        im = ax[a_, b_].imshow(maps[obj, s].T, origin='lower', extent=extent, cmap='magma', vmin=0, vmax=8)
        ax[a_, b_].set(title=f'{key} — {s.upper()}: S = {row.S_at_object:.1f} (null 98%: {row.null_98:.1f})', xticks=[], yticks=[])
        ax[a_, b_].title.set_fontsize(8)
        ax[a_, b_].invert_xaxis()                    # east to the left
plt.tight_layout()

S = res.set_index(['object', 'survey']).S_at_object
D = res.set_index(['object', 'survey']).detected
# Known outcomes: Leo IV was discovered in SDSS, Virgo I and Boötes IV in HSC, and Virgo I is too faint for SDSS
assert D['leo_4', 'SDSS'] and D['virgo_1', 'HSC'] and D['bootes_4', 'HSC'] and not D['virgo_1', 'SDSS']

**Figure 12.2.** Search maps for Leo IV, Virgo I and Boötes IV (rows) in SDSS, PS1, the Legacy Surveys and HSC (columns):
in each 1′ pixel within 0.74° of the catalogued position (the panel centre; east to the left), the largest significance
S over distance moduli 18.5–22.5 and kernels of 1′, 2′ and 4′, computed from that survey's point sources in its own CMD
masks; S = 0 where there is no overdensity. Colour scale magma from S = 0 (black) to 8 (white, saturated above; no colour
bar). Each title gives S within 3′ of the catalogued position and the 98th percentile of the maxima of 50 null patches
of the same survey: Leo IV 5.2 / 4.8 / 13.1 / 21.8, Virgo I 1.9 / 2.3 / 5.3 / 5.6, Boötes IV 3.9 / 4.1 / 3.9 / 13.3 (SDSS /
PS1 / LS / HSC), against null thresholds of 4.6–5.1. Leo IV and Boötes IV become bright central peaks in the deep data;
in the Virgo I patch a stronger off-centre peak (south-east of centre) dominates the Legacy Surveys and HSC maps, and in
the Boötes IV HSC map a second peak lies south-east of the satellite.

Read the table row by row:

- **Leo IV** ($M_V = -5$, 151 kpc), discovered in SDSS, is only marginally above the SDSS null here and below the PS1
  null (whose strict PSF − Kron cut loses faint stars); it is obvious in the Legacy Surveys (S ≈ 13) and HSC (S ≈ 22),
  which reach its turn-off region and horizontal branch with far more stars.
- **Virgo I** ($M_V \approx -0.9$) is invisible in SDSS and PS1, and only just above the null in the Legacy Surveys and
  HSC: it is one of the faintest known satellites.
- **Boötes IV** (~210 kpc) is found only in HSC, where it was discovered.

Two warnings. First, the column `max_elsewhere`: in several patches another peak is *stronger* than the satellite. In
the Virgo I patch the strongest peak (S ≈ 17 in the Legacy Surveys) lies 0.5′ from NGC 4030, a bright spiral galaxy that
the pipelines split into many point-like pieces — the same false positive as UGC 6903 in Chapter 9. A blind search would
rank such objects first; vetting is essential at every depth. Second, the null thresholds (≈ 4.6–5.1) come from uniformly
placed stars and are optimistic for the same reason (Chapter 9).

Depth helps, but it is not the whole story: image quality and the star–galaxy classifier decide how many faint galaxies
contaminate the "stars", and each survey needs its own null calibration.

## Summary

- The same search, adapted to each survey's photometry, errors and isochrone system, gives very different answers.
- Faint satellites at ~100–200 kpc need HSC-like depth *and* good star–galaxy separation; Virgo I is barely found even in HSC.
- Shredded bright galaxies produce stronger peaks than real satellites in deep data too.
- Judge each survey against its own null distribution; never compare raw significances across surveys.